# 07 — Sequential scenario arrays for modern Brightway

**Audience:** Modern Brightway users comparing the same functional unit across several Premise scenarios without writing one database per scenario.

**Prerequisites:** `premise[bw25]`, `bw2data >= 4`, `bw_processing >= 1`, ecoinvent 3.12 cutoff and biosphere databases, an LCIA method, and `PREMISE_KEY`.

**Learning goals:** write one union database and scenario-array ZIP, load the ZIP after the database datapackages, iterate deterministically, and validate the original state and wraparound.


## Outline

1. Validate databases and the LCIA method.
2. Build three IMAGE scenarios.
3. Export the union database and ZIP.
4. Calculate and validate scenario scores.


In [ ]:
import os

import bw2calc as bc
import bw2data as bd
import numpy as np
import pandas as pd

from premise import NewDatabase
from premise.utils import create_scenario_list

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
DATABASE_NAME = "premise-image-2050-scenario-array"
PREMISE_KEY = os.environ.get("PREMISE_KEY")
METHOD = (
    "ecoinvent-3.12",
    "EF v3.1",
    "climate change",
    "global warming potential (GWP100)",
)

if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY before running this tutorial.")

bd.projects.set_current(PROJECT)
missing = [
    name for name in (SOURCE_DATABASE, BIOSPHERE_DATABASE) if name not in bd.databases
]
if missing:
    raise ValueError(f"Missing Brightway databases: {missing}")
if METHOD not in bd.methods:
    raise ValueError(f"Missing LCIA method: {METHOD}")


## 1. Define and build the scenarios

Scenario order becomes array-column order after `original`. Labels must be unique.


In [ ]:
SCENARIOS = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050},
    {"model": "image", "pathway": "SSP2-M", "year": 2050},
    {"model": "image", "pathway": "SSP3-H", "year": 2050},
]
scenario_labels = ["original", *create_scenario_list(SCENARIOS)]

ndb = NewDatabase(
    scenarios=[scenario.copy() for scenario in SCENARIOS],
    source_db=SOURCE_DATABASE,
    source_version="3.12",
    source_type="brightway",
    system_model="cutoff",
    biosphere_name=BIOSPHERE_DATABASE,
    key=PREMISE_KEY,
)
ndb.update()


## 2. Write the database and array ZIP

With no `filepath`, the ZIP is written below `export/scenario arrays/`. It contains only coordinates that change in at least one scenario.


In [ ]:
array_path = ndb.write_scenario_array_db_to_brightway(
    name=DATABASE_NAME,
)
print(f"Database: {DATABASE_NAME}")
print(f"Scenario arrays: {array_path}")


## 3. Match activities exactly

Match activity name, reference product, and location together. Similar names in multiple locations are common.


In [ ]:
def find_activity(database, *, name, product, location):
    matches = [
        activity
        for activity in database
        if activity.get("name") == name
        and activity.get("reference product") == product
        and activity.get("location") == location
    ]
    if len(matches) != 1:
        keys = [activity.key for activity in matches[:20]]
        raise ValueError(
            f"Expected one match for {name!r}, {product!r}, {location!r}; "
            f"found {len(matches)}: {keys}"
        )
    return matches[0]


## 4. Calculate and validate scores

Append the ZIP **after** `data_objs`, so its changing values override the union database. Array selection is deterministic enumeration, not Monte Carlo sampling.


In [ ]:
def calculate_scenario_scores(activity):
    demand, data_objs, remapping = bd.prepare_lca_inputs(
        {activity: 1},
        method=METHOD,
    )
    lca = bc.LCA(
        demand,
        data_objs=[*data_objs, array_path],
        remapping_dicts=remapping,
        use_arrays=True,
        use_distributions=False,
    )
    lca.lci()
    lca.lcia()

    scores = [float(lca.score)]
    for _ in ndb.scenarios:
        next(lca)
        scores.append(float(lca.score))

    next(lca)
    if not np.isclose(lca.score, scores[0], rtol=1e-6, atol=1e-9):
        raise AssertionError("Scenario selection did not wrap to original.")

    base_lca = bc.LCA(
        demand,
        data_objs=data_objs,
        remapping_dicts=remapping,
        use_arrays=False,
        use_distributions=False,
    )
    base_lca.lci()
    base_lca.lcia()
    if not np.isclose(base_lca.score, scores[0], rtol=1e-6, atol=1e-9):
        raise AssertionError("Original array state differs from the base database.")

    return scores


In [ ]:
ACTIVITY_SELECTORS = [
    {
        "name": "market for electricity, low voltage",
        "product": "electricity, low voltage",
        "location": "CH",
    },
    {
        "name": "market for steel, low-alloyed",
        "product": "steel, low-alloyed",
        "location": "GLO",
    },
]

database = bd.Database(DATABASE_NAME)
rows = []
for selector in ACTIVITY_SELECTORS:
    activity = find_activity(database, **selector)
    for label, score in zip(
        scenario_labels,
        calculate_scenario_scores(activity),
    ):
        rows.append(
            {
                "activity": activity.get("name"),
                "location": activity.get("location"),
                "scenario": label,
                "score": score,
            }
        )

scores = pd.DataFrame(rows)
scores.pivot(
    index=["activity", "location"],
    columns="scenario",
    values="score",
).reindex(columns=scenario_labels)


## Reference result

The table below records an end-to-end validation with Premise 2.4.9.2, ecoinvent 3.12 cutoff, and EF v3.1 GWP100. Scores are in kg CO₂-eq per activity unit and may change with Premise, IAM files, inventories, or LCIA methods.

| Functional unit | original | IMAGE SSP1-L 2050 | IMAGE SSP2-M 2050 | IMAGE SSP3-H 2050 |
|---|---:|---:|---:|---:|
| Swiss low-voltage electricity, 1 kWh | 0.031382 | 0.037540 | 0.037347 | 0.117412 |
| Swiss low-sulfur diesel, 1 kg | 0.970115 | 0.787928 | 0.815387 | 0.827007 |
| Global low-alloyed steel, 1 kg | 2.027076 | 1.266054 | 1.549705 | 1.667573 |


## Lifecycle and pitfalls

- The ZIP contains node IDs from the active project. Regenerate it after moving, deleting, or rewriting the union database.
- Append the ZIP last and advance the joint technosphere/biosphere package as one unit.
- One extra `next(lca)` after the final scenario returns to `original`.
- Existing uncertainty metadata is outside the scenario-array package.

## Exercise

Score one additional activity without rebuilding the database or ZIP.


In [ ]:
exercise_selector = {
    "name": "market for diesel, low-sulfur",
    "product": "diesel, low-sulfur",
    "location": "CH",
}
exercise_activity = find_activity(database, **exercise_selector)
dict(zip(scenario_labels, calculate_scenario_scores(exercise_activity)))
